# Global soil-moisture pipeline

This notebook is the interactive entry point for the complete global pipeline. Run the cells in order, or rerun only the stage you are working on.

The pipeline is split into these independently inspectable stages:

1. Audit the ISMN archive and build the station catalog.
2. Select the compatible cohort and create network-disjoint splits.
3. Build the good-quality station/date label manifest.
4. Plan Earth Engine exports, optionally submit them, and inspect the task manifest.
5. After Earth Engine feature files exist, validate the feature manifest and pack tensors.

Earth Engine submission is disabled by default. Set `SUBMIT_EE = True` only in the export cell after confirming the generated task manifest.

In [1]:
from pathlib import Path
from datetime import date
import csv
import json
import sys
ROOT = Path.cwd()
if not (ROOT / "global_data_contract.py").exists():
    ROOT = Path("/home/mayur/Code/UG_SM")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from build_global_ismn_catalog import build_catalog, write_outputs
from build_global_label_dates import build_label_dates
from build_global_splits import _read_csv, assign_network_splits, select_cohort, write_manifest as write_split_manifest
from build_global_tensors import pack_tensors, read_feature_manifest
from gee_global_export import (
        adaptive_tiles,
        build_tasks,
        collapse_resolution_cells,
        read_stations,
        submit_tasks,
        write_manifest as write_ee_manifest,
    )
from global_data_contract import GlobalDataContract

## 1. Configure the run

Edit values in this cell instead of passing command-line arguments. Paths are relative to the repository root.

In [2]:
ROOT = Path.cwd()
if not (ROOT / "global_data_contract.py").exists():
    ROOT = Path("/home/mayur/Code/UG_SM")

archive_candidates = sorted((ROOT / "Data" / "Global").glob("*.zip"))
ARCHIVE = archive_candidates[0] if len(archive_candidates) == 1 else None
OUTPUT_DIR = ROOT / "global_pipeline_output"

MINIMUM_GOOD_ROWS = 1
SPLIT_SEED = 20261003
VALIDATION_FRACTION = 0.2
TEST_FRACTION = 0.2

EXPORT_START = date(2015, 4, 1)
EXPORT_END = date(2026, 10, 3)
EXPORT_YEAR = 2015  # Set an integer such as 2025 to plan one year only.
DRIVE_FOLDER = "global_ismn_exports"
MAX_STATIONS_PER_TILE = 100
TILE_SCALE = 16
S1_SPECKLE_RADIUS = 0
EE_PROJECT = "dogwood-concept-493414-v9"  # Required for submission when Earth Engine is not initialized.
SUBMIT_EE = True
MAX_EE_TASKS = None  # Set an integer for a safe trial submission.

FEATURE_MANIFEST = None  # Path to the CSV created after Earth Engine exports finish.

if ARCHIVE is None:
    raise FileNotFoundError("Set ARCHIVE to the ISMN .zip file; none or multiple archives were found.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CATALOG_CSV = OUTPUT_DIR / "global_station_catalog.csv"
CATALOG_JSON = OUTPUT_DIR / "global_station_catalog_summary.json"
SPLIT_CSV = OUTPUT_DIR / "global_split_manifest.csv"
SPLIT_JSON = OUTPUT_DIR / "global_split_summary.json"
LABEL_DATES_CSV = OUTPUT_DIR / "global_label_dates.csv"
EE_OUTPUT_DIR = OUTPUT_DIR / "global_gee_exports"
TENSOR_OUTPUT_DIR = OUTPUT_DIR / "global_tensors"

print(f"Repository: {ROOT}")
print(f"Archive: {ARCHIVE}")
print(f"Outputs: {OUTPUT_DIR}")

Repository: /home/mayur/Code/UG_SM
Archive: /home/mayur/Code/UG_SM/Data/Global/Data_separate_files_header_20150401_20261003_13386_pCU5_20261003.zip
Outputs: /home/mayur/Code/UG_SM/global_pipeline_output


## 2. Build and inspect the ISMN station catalog

This scans station headers and observations, counts good-quality rows, and records depth compatibility.

In [3]:
catalog_records = build_catalog(ARCHIVE, scan_observations=True) # type: ignore
if not catalog_records:
    raise ValueError("No .stm station files were found in the archive.")
write_outputs(catalog_records, CATALOG_CSV, CATALOG_JSON)

catalog = pd.read_csv(CATALOG_CSV)
display(catalog.head())
display(catalog.groupby("depth_class", dropna=False).size().rename("station_files"))
print(f"Catalog rows: {len(catalog):,}")

,source_file,network,network_name,station,latitude,longitude,elevation_m,depth_from_m,depth_to_m,sensor,depth_class,exclusion_reason,static_variables_file,observation_rows,good_quality_rows,malformed_rows,first_observation_date,last_observation_date
0,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.53250,2.66040,208.0,0.05,0.05,Campbell-CS616,near_surface,NaN,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,32904,32727,0,2015-04-01,2018-12-31
1,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.53250,2.66040,208.0,0.05,0.05,Campbell-CS616,near_surface,NaN,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,32904,32756,0,2015-04-01,2018-12-31
2,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,AMMA-CATCH,AMMA-CATCH,Belefoungou-Mid,9.79506,1.70994,414.0,0.05,0.05,Campbell-CS616,near_surface,NaN,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,22620,22524,0,2015-04-01,2017-11-10
3,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,AMMA-CATCH,AMMA-CATCH,Belefoungou-Mid,9.79506,1.70994,414.0,0.10,0.10,Campbell-CS616,fallback_deeper,NaN,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,22620,22600,0,2015-04-01,2017-11-10
4,AMMA-CATCH/Belefoungou-Top/AMMA-CATCH_AMMA-CAT...,AMMA-CATCH,AMMA-CATCH,Belefoungou-Top,9.78986,1.70994,414.0,0.05,0.05,Campbell-CS616,near_surface,NaN,AMMA-CATCH/Belefoungou-Top/AMMA-CATCH_AMMA-CAT...,23072,22996,0,2015-04-01,2017-12-31


depth_class
excluded            338
fallback_deeper    2424
near_surface       1578
Name: station_files, dtype: int64

Catalog rows: 4,340


## 3. Select the cohort and assign network-disjoint splits

In [4]:
catalog_rows = _read_csv(CATALOG_CSV)
cohort_rows = select_cohort(
    catalog_rows,
    minimum_good_quality_rows=MINIMUM_GOOD_ROWS,
)
split_rows = assign_network_splits(
    cohort_rows,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
    seed=SPLIT_SEED,
)
write_split_manifest(split_rows, SPLIT_CSV, SPLIT_JSON)

split_manifest = pd.read_csv(SPLIT_CSV)
display(split_manifest.groupby("split").agg(station_files=("station", "size"), networks=("network", "nunique")))
display(split_manifest.head())

,station_files,networks
split,,
test,599,11
train,3021,35
validation,328,11


,source_file,network,network_name,station,latitude,longitude,elevation_m,depth_from_m,depth_to_m,sensor,...,exclusion_reason,static_variables_file,observation_rows,good_quality_rows,malformed_rows,first_observation_date,last_observation_date,cohort_eligible,split,split_seed
0,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.53250,2.66040,208.0,0.05,0.05,Campbell-CS616,...,NaN,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,32904,32727,0,2015-04-01,2018-12-31,True,train,20261003
1,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.53250,2.66040,208.0,0.05,0.05,Campbell-CS616,...,NaN,AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_B...,32904,32756,0,2015-04-01,2018-12-31,True,train,20261003
2,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,AMMA-CATCH,AMMA-CATCH,Belefoungou-Mid,9.79506,1.70994,414.0,0.05,0.05,Campbell-CS616,...,NaN,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,22620,22524,0,2015-04-01,2017-11-10,True,train,20261003
3,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,AMMA-CATCH,AMMA-CATCH,Belefoungou-Mid,9.79506,1.70994,414.0,0.10,0.10,Campbell-CS616,...,NaN,AMMA-CATCH/Belefoungou-Mid/AMMA-CATCH_AMMA-CAT...,22620,22600,0,2015-04-01,2017-11-10,True,train,20261003
4,AMMA-CATCH/Belefoungou-Top/AMMA-CATCH_AMMA-CAT...,AMMA-CATCH,AMMA-CATCH,Belefoungou-Top,9.78986,1.70994,414.0,0.05,0.05,Campbell-CS616,...,NaN,AMMA-CATCH/Belefoungou-Top/AMMA-CATCH_AMMA-CAT...,23072,22996,0,2015-04-01,2017-12-31,True,train,20261003


## 4. Build the label-date manifest

Only compatible depths and observations with quality flag `G` are included.

In [5]:
label_rows = build_label_dates(ARCHIVE) # type: ignore
if not label_rows:
    raise ValueError("No good-quality compatible label dates were found.")
with LABEL_DATES_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(label_rows[0]))
    writer.writeheader()
    writer.writerows(label_rows)

label_dates = pd.DataFrame(label_rows)
display(label_dates.head())
print(f"Label rows: {len(label_dates):,}; dates: {label_dates['sample_date'].nunique():,}")

,cell_id,sample_date,station_id,depth_class
0,x+000005_y+001134,2019-12-08,"TAHMO__Bimbilla_SHS,_Bimbilla",fallback_deeper
1,x+000005_y+001134,2019-12-09,"TAHMO__Bimbilla_SHS,_Bimbilla",fallback_deeper
2,x+000005_y+001134,2019-12-10,"TAHMO__Bimbilla_SHS,_Bimbilla",fallback_deeper
3,x+000005_y+001134,2019-12-11,"TAHMO__Bimbilla_SHS,_Bimbilla",fallback_deeper
4,x+000005_y+001134,2019-12-12,"TAHMO__Bimbilla_SHS,_Bimbilla",fallback_deeper


Label rows: 4,095,965; dates: 4,204


## 5. Plan Earth Engine exports

This cell is safe to run without the Earth Engine package. It creates deterministic spatial tiles, yearly task definitions, and a collision map. Review `export_tasks.csv` before enabling submission.

In [6]:
stations = read_stations(SPLIT_CSV)
representative_stations, collision_map = collapse_resolution_cells(stations)
tiles = adaptive_tiles(representative_stations, MAX_STATIONS_PER_TILE)

label_dates_by_cell = {}
for row in label_rows:
    label_dates_by_cell.setdefault(row["cell_id"], set()).add(row["sample_date"])

tasks = build_tasks(
    tiles,
    EXPORT_START,
    EXPORT_END,
    DRIVE_FOLDER,
    years=1,
    only_year=EXPORT_YEAR,
    label_dates=label_dates_by_cell,
    include_static=True,
)
write_ee_manifest(
    tasks,
    tiles,
    EE_OUTPUT_DIR,
    start=EXPORT_START,
    end=EXPORT_END,
    maximum=MAX_STATIONS_PER_TILE,
    collision_map=collision_map,
)

task_manifest = pd.DataFrame([task.__dict__ for task in tasks])
display(task_manifest.groupby("export_kind").size().rename("tasks"))
display(task_manifest.head())
print(f"Tiles: {len(tiles)}; tasks: {len(tasks)}")

export_kind
daily     17
static    17
Name: tasks, dtype: int64

,task_id,tile_id,station_count,start_date,end_date,description,drive_folder,file_prefix,file_format,compressed,export_kind,label_date_count
0,tile_0000_static,tile_0000,96,2015-04-01,2026-10-03,global_ismn_tile_0000_static,global_ismn_exports,global_ismn_tile_0000_static,TFRecord,True,static,0
1,tile_0000_20150401_20160101,tile_0000,96,2015-04-01,2016-01-01,global_ismn_tile_0000_20150401_20160101,global_ismn_exports,global_ismn_tile_0000_20150401_20160101,TFRecord,True,daily,275
2,tile_0001_static,tile_0001,96,2015-04-01,2026-10-03,global_ismn_tile_0001_static,global_ismn_exports,global_ismn_tile_0001_static,TFRecord,True,static,0
3,tile_0001_20150401_20160101,tile_0001,96,2015-04-01,2016-01-01,global_ismn_tile_0001_20150401_20160101,global_ismn_exports,global_ismn_tile_0001_20150401_20160101,TFRecord,True,daily,275
4,tile_0002_static,tile_0002,96,2015-04-01,2026-10-03,global_ismn_tile_0002_static,global_ismn_exports,global_ismn_tile_0002_static,TFRecord,True,static,0


Tiles: 17; tasks: 34


### Submit Earth Engine tasks

This follows the same initialization pattern as `CNN_Spain.ipynb`: authenticate, initialize Earth Engine, create each `Export.table.toDrive` task, and start it. Submission starts Earth Engine tasks and may incur quotas or processing costs.

In [7]:
if SUBMIT_EE:
    submit_tasks(
        tasks,
        tiles,
        project=EE_PROJECT,
        max_tasks=MAX_EE_TASKS,
        authenticate=True,
        label_dates=label_dates_by_cell,
        tile_scale=TILE_SCALE,
        s1_speckle_radius=S1_SPECKLE_RADIUS,
    )
else:
    print("Earth Engine submission is disabled. Set SUBMIT_EE = True after reviewing the task manifest.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Started tile_0000_static: PBGQSRX4FRA6M2XPI2NYI53Z
Started tile_0000_20150401_20160101: MUGPAGQAW6TXGZIZZU7SCK7C
Started tile_0001_static: 324IO7KXA7IHF5OCGUFVNLRM
Started tile_0001_20150401_20160101: BHORUOVG2FEIMQ7DKLIDAAYW
Started tile_0002_static: EKTY2DFBFTVGHM4TYLQKRASX
Started tile_0002_20150401_20160101: EXZHYFGP45OTZ4ZM6SIX2EAZ
Started tile_0003_static: NV2GLADPCRAY5DEYXY5SQ3EY
Started tile_0003_20150401_20160101: 7AA5QWBXAWKEQJHRMRLNCN6E
Started tile_0004_static: F6DVYOPXY7DC733MXBRR5XYS
Started tile_0004_20150401_20160101: YRIMCWG435RLJNGEGQ356QYX
Started tile_0005_static: 65UCG5JRBHWMPMQVQCXZZZIR
Started tile_0005_20150401_20160101: LQSXPJWUNM24BG5BOXTKM22Z
Started tile_0006_static: 52LUORPZQBLSQQPLOPFS2PNQ
Started tile_0006_20150401_20160101: 6J4SMD564NIWXOQGIKVAPFMX
Started tile_0007_static: 4JQZQOEG2324N6CNNWPYCYG7
Started tile_0007_20150401_20160101: 2BS2OMT47KHPBTIIN2W5NYWQ
Started tile_0008_static: ZLWUSMRCYMLRKJ7RP5GCFHJV
Started tile_0008_20150401_20160101: A26WC33Y

## 6. Pack model-ready tensors

Run this stage after the Earth Engine exports have been downloaded and a station-date feature manifest has been created. The manifest must contain `station_id`, `network`, `sample_date`, `split`, `feature_path`, `target`, and `anchor`.

In [ ]:
if FEATURE_MANIFEST is None:
    print("Tensor packing is paused. Set FEATURE_MANIFEST to the feature CSV and rerun this cell.")
else:
    FEATURE_MANIFEST = Path(FEATURE_MANIFEST)
    feature_rows = read_feature_manifest(FEATURE_MANIFEST)
    packed_paths = pack_tensors(
        feature_rows,
        manifest_dir=FEATURE_MANIFEST.parent,
        output_dir=TENSOR_OUTPUT_DIR,
    )
    print(json.dumps({split: str(path) for split, path in packed_paths.items()}, indent=2))
    for split, path in packed_paths.items():
        with __import__("numpy").load(path) as data:
            print(split, {name: array.shape for name, array in data.items()})

## Contract

The notebook uses the same versioned contract as the library modules.

In [ ]:
GlobalDataContract().as_dict()